In [12]:
import numpy as np

def global_func_linear(seq1, seq2, match_score=3, mismatch_score=-3, gap=-4):

    m, n = len(seq1), len(seq2)
    F = np.zeros((m + 1, n + 1), dtype=int)
    traceback = np.zeros((m + 1, n + 1), dtype=int)
    
    # заполним первые строку и столбец в F 
    for i in range(1, m + 1):
        F[i][0] = i * gap
        traceback[i][0] = 1 
        
    for j in range(1, n + 1):
        F[0][j] = j * gap
        traceback[0][j] = 2 
    
    # полностью заполним F с учетом правил, рекуррентно
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i-1] == seq2[j-1]:
                diagonal_score = F[i-1][j-1] + match_score
            else:
                diagonal_score = F[i-1][j-1] + mismatch_score
            
            up_score = F[i-1][j] + gap
            left_score = F[i][j-1] + gap
            
            scores = [diagonal_score, up_score, left_score]
            max_score = max(scores)
            F[i][j] = max_score
            traceback[i][j] = scores.index(max_score)
    
    # обратный ход
    aligned_seq1 = []
    aligned_seq2 = []
    i, j = m, n
    
    while i > 0 or j > 0:
        if traceback[i][j] == 0:
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append(seq2[j-1])
            i -= 1
            j -= 1
        elif traceback[i][j] == 1:
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append('-')
            i -= 1
        else:
            aligned_seq1.append('-')
            aligned_seq2.append(seq2[j-1])
            j -= 1
    
    aligned_seq1 = ''.join(reversed(aligned_seq1))
    aligned_seq2 = ''.join(reversed(aligned_seq2))
    
    score = F[m][n]
    
    return score, F, aligned_seq1, aligned_seq2


def global_func_affine(seq1, seq2, match_score=3, mismatch_score=-3, open_penalty=-10, extend_penalty=-1):

    m, n = len(seq1), len(seq2)
    
    # три матрицы для трех состояний
    M = np.zeros((m + 1, n + 1), dtype=int)  # заканчивается совпадением или несовпадением
    X = np.zeros((m + 1, n + 1), dtype=int)  # заканчивается гэпом в seq2
    Y = np.zeros((m + 1, n + 1), dtype=int)  # заканчивается гэпом в seq1
    
    traceback = np.zeros((m + 1, n + 1, 3), dtype=int)
    
    # инициализация с бесконечностью для невозможных состояний)
    for i in range(1, m + 1):
        M[i][0] = -10**9
        X[i][0] = open_penalty + (i-1) * extend_penalty
        Y[i][0] = -10**9
        
    for j in range(1, n + 1):
        M[0][j] = -10**9
        X[0][j] = -10**9
        Y[0][j] = open_penalty + (j-1) * extend_penalty
    
    M[0][0] = 0
    X[0][0] = -10**9
    Y[0][0] = -10**9
    
    # заполняем матрицы
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i-1] == seq2[j-1]:
                s = match_score #мэтч 
            else:
                s = mismatch_score # мисмэтч
             
            candidates_M = [
                M[i-1][j-1] + s,
                X[i-1][j-1] + s,
                Y[i-1][j-1] + s
            ]
            M[i][j] = max(candidates_M)
            traceback[i][j][0] = candidates_M.index(M[i][j])
            
            candidates_X = [
                M[i-1][j] + open_penalty,      
                X[i-1][j] + extend_penalty,    
                Y[i-1][j] + open_penalty      
            ]
            X[i][j] = max(candidates_X)
            traceback[i][j][1] = candidates_X.index(X[i][j])
            
            candidates_Y = [
                M[i][j-1] + open_penalty,      
                X[i][j-1] + open_penalty,      
                Y[i][j-1] + extend_penalty   
            ]
            Y[i][j] = max(candidates_Y)
            traceback[i][j][2] = candidates_Y.index(Y[i][j])
    
    # score
    final_score = max(M[m][n], X[m][n], Y[m][n])
    final_states = [M[m][n], X[m][n], Y[m][n]]
    final_state = final_states.index(final_score)
    
    aligned_seq1 = []
    aligned_seq2 = []
    i, j = m, n
    state = final_state 
    
    while i > 0 or j > 0:
        if state == 0:  
            prev = traceback[i][j][0]
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append(seq2[j-1])
            i -= 1
            j -= 1
            state = prev
        elif state == 1:  # гэп в seq2
            prev = traceback[i][j][1]
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append('-')
            i -= 1
            state = prev
        else:  # гэп в seq1
            prev = traceback[i][j][2]
            aligned_seq1.append('-')
            aligned_seq2.append(seq2[j-1])
            j -= 1
            state = prev
    
    aligned_seq1 = ''.join(reversed(aligned_seq1))
    aligned_seq2 = ''.join(reversed(aligned_seq2))
    
    return final_score, (M, X, Y), aligned_seq1, aligned_seq2


X = "ATGCAGCAGCAGCCA"
Y = "ATATAT"

# линейный штраф
score_linear, matrix_linear, aligned_X_linear, aligned_Y_linear = global_func_linear( X, Y, match_score=3, mismatch_score=-3, gap=-4)

print(f"Матрица Нидлмана-Вунша с линейным штрафом:")
print(matrix_linear)
print(f"\nScore: {score_linear}")
print(f"Выравнивание:")
print(f"X: {aligned_X_linear}")
print(f"Y: {aligned_Y_linear}")

# аффинный штраф
score_affine, matrices_affine, aligned_X_affine, aligned_Y_affine = global_func_affine( X, Y, match_score=3, mismatch_score=-3, open_penalty=-10, extend_penalty=-1)
M, X_mat, Y_mat = matrices_affine
print("Матрица M (совпадения/несовпадения):")
print(M)
print("\nМатрица X (гэпы в Y):")
print(X_mat)
print("\nМатрица Y (гэпы в X):")
print(Y_mat)
print(f"\nScore аффинный: {score_affine}")
print(f"Выравнивание:")
print(f"X: {aligned_X_affine}")
print(f"Y: {aligned_Y_affine}")
print("""
Аффинная модель лучше описывает биологические процессы, потому что в биологических последовательностях, таких как ДНК, РНК и белки, вставки и делеции часто затрагивают несколько нуклеотидов подряд и вероятность открытия нового гэпа значительно ниже, чем вероятность продолжения уже существующего. Аффинная модель отражает это разными штрафами. В эволюционном процессе длинные делеции возникают в результате одной мутации, поэтому штрафовать каждый нуклеотид одинаково, как делается в линейной модели, некорректно.
""")

Матрица Нидлмана-Вунша с линейным штрафом:
[[  0  -4  -8 -12 -16 -20 -24]
 [ -4   3  -1  -5  -9 -13 -17]
 [ -8  -1   6   2  -2  -6 -10]
 [-12  -5   2   3  -1  -5  -9]
 [-16  -9  -2  -1   0  -4  -8]
 [-20 -13  -6   1  -3   3  -1]
 [-24 -17 -10  -3  -2  -1   0]
 [-28 -21 -14  -7  -6  -5  -4]
 [-32 -25 -18 -11 -10  -3  -7]
 [-36 -29 -22 -15 -14  -7  -6]
 [-40 -33 -26 -19 -18 -11 -10]
 [-44 -37 -30 -23 -22 -15 -14]
 [-48 -41 -34 -27 -26 -19 -18]
 [-52 -45 -38 -31 -30 -23 -22]
 [-56 -49 -42 -35 -34 -27 -26]
 [-60 -53 -46 -39 -38 -31 -30]]

Score: -30
Выравнивание:
X: ATGCAGCAGCAGCCA
Y: AT-----A-TA---T
Матрица M (совпадения/несовпадения):
[[          0 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000
  -1000000000]
 [-1000000000           3         -13          -8         -15         -10
          -17]
 [-1000000000         -13           6         -10          -5         -12
           -7]
 [-1000000000         -14         -10           3          -7          -8
           -9]
 [-